# Exercise: An AI-Powered SQL Agent (Solution)

This is the worked solution for the SQL agent exercise. It builds an agent that answers questions about a real dataset by writing and running SQL, using the raw Groq API and the hand-written agentic loop from notebook 1. The data is the January 2026 NYC yellow taxi trips, queried through DuckDB.

## Learning Objectives

At the end of this exercise, you should be able to:

- Build a SQL agent from scratch that inspects a schema and runs queries through tools.
- Test an agent by asserting on its answer and on the order of its tool calls.
- Evaluate open-ended answers with a second model acting as a judge.
- Track the token usage and estimated cost of a run.

## Setup

We use the raw Groq client and DuckDB. `load_dotenv` reads `GROQ_API_KEY` from the local `.env` file. Every call uses the same model at `temperature=0` for repeatable output.

In [ ]:
import json
import os
import urllib.request
from collections.abc import Callable
from typing import TypedDict, cast

import duckdb
from dotenv import load_dotenv
from groq import Groq
from groq.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam

In [ ]:
load_dotenv()

client = Groq()
MODEL = "openai/gpt-oss-20b"

## Load the data

The dataset is one month of NYC yellow taxi trips as a Parquet file. We download it once (about 64 MB), then load it into a DuckDB table named `trips`. DuckDB reads Parquet directly, so no other database server is needed.

In [ ]:
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
)
PARQUET_FILE = "yellow_tripdata_2026-01.parquet"

if not os.path.exists(PARQUET_FILE):
    print("Downloading the taxi data (about 64 MB) ...")
    urllib.request.urlretrieve(DATA_URL, PARQUET_FILE)

con = duckdb.connect("taxi.db")
con.execute(f"CREATE TABLE IF NOT EXISTS trips AS SELECT * FROM '{PARQUET_FILE}'")
row_count_result = con.execute("SELECT COUNT(*) FROM trips").fetchone()
if row_count_result is None:
    raise RuntimeError("The row-count query returned no result.")
row_count = row_count_result[0]
row_count

The table holds 3,724,889 trips. That is far too much data to paste into a prompt, which is the point: the agent should query the data with SQL, not read it all. The model never sees the rows directly, only the schema and the results of the queries it chooses to run.

## The tools: inspect the schema and run a query

The agent gets two tools. `get_schema` returns the columns and their types, and `run_sql` runs a query and returns up to 50 result rows. The model cannot touch the database itself: it can only ask us to call one of these functions, and our code runs them.

In [ ]:
def get_schema() -> str:
    """Return the columns and types of the trips table."""
    rows = con.execute("DESCRIBE trips").fetchall()
    return "\n".join(f"{name}: {dtype}" for name, dtype, *_ in rows)


def run_sql(query: str) -> str:
    """Run a SQL query against the trips table and return up to 50 result rows."""
    return str(con.sql(query).limit(50))


def parse_tool_arguments(raw_arguments: str | None) -> dict[str, object]:
    """Parse a tool-call JSON object and reject other JSON values."""
    arguments = json.loads(raw_arguments or "{}")
    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a JSON object.")
    return arguments


tool_functions: dict[str, Callable[..., object]] = {
    "get_schema": get_schema,
    "run_sql": run_sql,
}

## Describe the tools to the model

As in notebook 1, we describe each tool to the model with a JSON schema: its name, what it does, and its arguments. `get_schema` takes no arguments; `run_sql` takes a single `query` string.

In [ ]:
tools: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Return the schema (columns and types) of the trips table. Call this before writing SQL.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a SQL query against the trips table and return the result rows.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A valid DuckDB SQL query.",
                    }
                },
                "required": ["query"],
            },
        },
    },
]

## The agent loop

This is the same loop as notebook 1, with two additions useful for testing: it records the order of tool calls, and it adds up the token usage the API reports on every call. The system prompt tells the agent to inspect the schema first, then run one query, then answer.

In [ ]:
SYSTEM_PROMPT = (
    "You are a SQL analyst for a DuckDB table named trips. "
    "Always call get_schema first to inspect the columns, then write and run one "
    "SQL query with run_sql, and finally answer the question using the query result. "
    "Be concise."
)


class AgentResult(TypedDict):
    answer: str
    tool_calls: list[str]
    prompt_tokens: int
    completion_tokens: int


def run_sql_agent(question: str) -> AgentResult:
    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tool_calls_made: list[str] = []
    prompt_tokens = completion_tokens = 0
    final_answer = ""

    while True:
        response = client.chat.completions.create(
            model=MODEL, temperature=0, messages=messages, tools=tools
        )
        usage = response.usage
        if usage is None:
            raise RuntimeError("Groq returned no token-usage data.")
        prompt_tokens += usage.prompt_tokens
        completion_tokens += usage.completion_tokens
        message = response.choices[0].message

        if not message.tool_calls:
            if message.content is None:
                raise RuntimeError("Groq returned no final answer.")
            final_answer = message.content
            messages.append({"role": "assistant", "content": final_answer})
            break

        messages.append(
            cast(ChatCompletionMessageParam, message.model_dump(exclude_none=True))
        )
        for call in message.tool_calls:
            arguments = parse_tool_arguments(call.function.arguments)
            tool_calls_made.append(call.function.name)
            result = tool_functions[call.function.name](**arguments)
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": str(result)}
            )

    return {
        "answer": final_answer,
        "tool_calls": tool_calls_made,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }

## Ask the agent a question

The agent now answers a question it has never seen the data for, by inspecting the schema and writing its own SQL.

In [ ]:
result = run_sql_agent("How many trips had more than 5 passengers?")
result["answer"]

The agent answers that there were 4,894 trips with more than 5 passengers. It reached that by calling `get_schema` to find the `passenger_count` column, then running a `COUNT(*)` query with a `WHERE passenger_count > 5` filter. We never told it the column name, it discovered it from the schema.

## Test 1: assert the answer is correct

The most direct test computes the true value straight from the database and checks that the agent's answer contains it. This catches wrong SQL and wrong arithmetic.

In [ ]:
expected_result = con.execute(
    "SELECT COUNT(*) FROM trips WHERE passenger_count > 5"
).fetchone()
if expected_result is None:
    raise RuntimeError("The validation query returned no result.")
expected = expected_result[0]

assert str(expected) in result["answer"] or f"{expected:,}" in result["answer"]
print(f"passed: the agent reported {expected:,} trips")

## Test 2: assert the tool-call order

A correct answer is not enough: we also want the agent to behave correctly. It should inspect the schema before it writes SQL, so it does not guess column names. We assert that the first tool call was `get_schema` and that `run_sql` was used.

In [ ]:
assert result["tool_calls"][0] == "get_schema"
assert "run_sql" in result["tool_calls"]
print("passed: tool order was", result["tool_calls"])

## Test 3: an LLM as a judge

Some answers cannot be checked with an exact string match, for example a one-sentence explanation. A second model can grade the answer against a natural-language criterion. The judge is told to reply with only `PASS` or `FAIL`.

In [ ]:
def judge(question: str, answer: str, criteria: str) -> str:
    """Ask a second model whether the answer meets a natural-language criterion."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": "You are a strict grader. Decide only whether the answer, as written, satisfies the stated criteria. Do not use outside knowledge and do not try to verify any facts or figures yourself: you have no access to the underlying data, so judge only whether the answer meets the criteria. Reply with only PASS or FAIL.",
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n"
                    f"Answer: {answer}\n"
                    f"Criteria: {criteria}\n"
                    "Reply with only PASS or FAIL."
                ),
            },
        ],
    )
    content = response.choices[0].message.content
    if content is None:
        raise RuntimeError("The grader returned no text response.")
    return content.strip()


judge(
    "How many trips had more than 5 passengers?",
    result["answer"],
    f"The answer states that the count is {expected}.",
)

The judge returns `PASS`, because the agent's answer does state the correct count. An LLM judge is useful when the target is a quality bar ("mentions the right figure", "is concise", "explains the reasoning") rather than an exact value, but treat its verdict as a signal, not a guarantee.

## Test 4: several scenarios

Real test suites cover more than one case. Each scenario pairs a question with the criterion its answer must meet. We run the agent on each and let the judge grade it, printing the verdict, the answer, and the tool order.

In [ ]:
scenarios = [
    {
        "question": "What is the average trip distance across all trips?",
        "criteria": "States an average distance of roughly 6.46 miles.",
    },
    {
        "question": "What is the total number of trips in the table?",
        "criteria": "States that there are about 3,724,889 trips.",
    },
]

for case in scenarios:
    run = run_sql_agent(case["question"])
    verdict = judge(case["question"], run["answer"], case["criteria"])
    print(f"[{verdict}] {case['question']}")
    print("   ->", run["answer"])
    print("   tools:", run["tool_calls"])

## Test 5: track the cost

Every response reports how many tokens it used, which we summed in the loop. Groq bills per token, so we can turn the counts into an estimated cost. The rates below are approximate and change over time, so treat the figure as a rough guide.

In [ ]:
INPUT_RATE = 0.075 / 1_000_000  # USD per input token (approximate)
OUTPUT_RATE = 0.30 / 1_000_000  # USD per output token (approximate)

cost = result["prompt_tokens"] * INPUT_RATE + result["completion_tokens"] * OUTPUT_RATE
{
    "prompt_tokens": result["prompt_tokens"],
    "completion_tokens": result["completion_tokens"],
    "estimated_usd": round(cost, 6),
}

One question used roughly 1,100 input tokens and under 100 output tokens, for an estimated cost well under a cent. Note that this estimate covers only the SQL agent's own calls (schema, query, answer), which are the calls `run_sql_agent` sums into `prompt_tokens` and `completion_tokens`. The `judge` call is not included, because `judge()` does not read its own `response.usage`. The schema and the tool results dominate the input tokens, which is worth remembering: the more you feed back into the context, the more each step costs.

## Summary

In this exercise you:

- Built a SQL agent from scratch that inspects a schema and runs queries through two tools.
- Asserted on both the agent's answer and the order of its tool calls.
- Used a second model as a judge to grade an answer against a natural-language criterion.
- Tracked token usage and estimated the cost of a run.

The agent is the same hand-written loop from notebook 1, pointed at real tools over a real database. Everything the agent "knows" about the data, it discovered by calling `get_schema` and `run_sql`, and everything it remembers is in the `messages` list we carried forward.

## References & Further Reading

- [**Groq Tool Use**](https://console.groq.com/docs/tool-use): Function calling and the tool-call loop.
- [**DuckDB Documentation**](https://duckdb.org/docs/): Querying Parquet and running SQL in-process.
- [**NYC TLC Trip Record Data**](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): The source of the taxi dataset.
- [**Groq Console**](https://console.groq.com/playground): Create a free API key and try the model used here.